# 리포트 10-2 — 결론이 무엇에 기대고 있나 — 강건성과 하드웨어

> 표적 모형·수신 소자·장비를 바꿔 넣어 **결론이 어디서 흔들리는지** 본다.

이 편은 [리포트 10 «결과 — 얼마나 멀리서 보이나»](10_results.ipynb) 의 **별편**이다 — 그 권 물음의 심화·지원·변주이지, 그림 무게로 나눈 분권이 아니다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 ⭐ | σ 를 곱하기 전에 이미 세 파형의 순서를 정하는 축이 있다 | `_parts/64_sigma-free-axis.ipynb` |
| 2 | 평판·큐브·우리 격자를 같은 동작점에서 갈아끼우면 요구 이득이 이만큼 달라진다 | `_parts/65_target-model-swap.ipynb` |
| 3 | 코히어런트 배열이득은 10log₁₀N 상한에 -0.11~+0.47 dB 로 붙는다 | `_parts/66_rx-elements.ipynb` |
| 4 | X410 의 12-bit ADC 동적범위가 직접파 제거의 천장이다 | `_parts/67_hardware.ipynb` |
| 5 | 교정된 절대 σ 를 만드는 조건은 여섯 항목이 전부다 | `_parts/68_sigma-checklist.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열세 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. σ 를 곱하기 전에 이미 세 파형의 순서를 정하는 축이 있다



> ### 한 일
> **같은 표적·같은 기하·같은 σ 로 묶은 벤치에서 Pd = 0.5 에 필요한 출력 SNR 을 재고, 두 점유 등급 사이의 요구 SNR 차를 표준마다 dB 로 적었다.**

### 결과
1. WiFi 는 상시 기준(등급 1)에서 11.87 dB [^1], 세션 기준(등급 3)에서 11.66 dB [^2] 라 두 등급의 차가 +0.21 dB [^3] 다.
2. ⛔ 그 차는 기준신호 대역의 몫이 아니다 — W1·W3 의 기준신호 대역은 76.56 MHz [^4] ↔ 76.56 MHz [^5] 로 같고, 데이터 점유가 9.1% [^6] → 89.1% [^7] 로 갈린다.
3. LTE 의 차는 +0.08 dB [^8], 5G 는 +3.82 dB [^9] 다 — 셋 다 K = 6000 [^10] 몬테카를로 SNR50 두 값의 차이고, 이 원장의 표준편차 0.043 dB [^11] 는 벤치 전 모드·전 N 의 최댓값이라(`src/make_report05_results.py:290`) 모드 쌍별 유의성은 이 원장 밖에 있다.
4. 이 축은 σ 와 무관하다 — 표적·기하·σ 를 한 값으로 묶었으므로 여기서 읽는 것은 파형 축 하나의 상대 비교다.
5. 벤치 배치는 단일 반송파 3.5 GHz [^12] · 바이스태틱 거리 22.3 m [^13] · 고정 σ -27.58 dBsm [^14] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 무엇을 고정했나 | 9모드 전부에 단일 반송파·단일 기하·단일 σ 를 쓴다 — `src/experiment_x410.py:101` |
| 무엇을 읽나 | 기준신호 대역 · 프레임 수 · 데이터 점유가 함께 정하는 파형 축 하나다. 그래서 이 순서는 σ 를 곱하기 앞에서 이미 정해진다 |
| 상관 기준 규약 | 기지 기준신호가 아니라 그 모드의 **송신파형 전체**를 상관 기준으로 쓴다 — `src/experiment_detection.py:145` 의 `ref_cpi = np.tile(block, b * M)`, 규약은 같은 파일 `:124` 주석에 «full-waveform capture 상한» 으로 적혀 있다 |
| 다른 절과의 관계 | 이 스윕은 자유공간 배치와 **다른 배치**에서 돈다 — 절대 SNR 을 그 배치의 거리와 같은 축에 놓는 일은 다음 단계에 있다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_detection.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/detection_rx_sweep.json`, `outputs/report05_derived.json` |
| 소요 | σ 격자 · 검지거리 4단계 · 검증 · 스윕을 합쳐 7.8 h [^15] (GPU 2 [^16]장). 이 빌더 자신은 CPU 수 초다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [리포트 7 절 3 «여섯 항목은 닫힌형이고»](07_illuminators.ipynb) | 점유 등급과 기준신호 대역의 대가 |

---


## 기준신호 대역과 점유 등급이 만드는 축

같은 표적·같은 기하·같은 검출기에서 Pd = 0.5 에 필요한 출력 SNR 은 기준신호 대역 · 프레임 수 · 데이터 점유가 정한다. 이 축은 σ 와 무관하게 세 파형의 순서를 정한다.

⚠ 상시(등급 1) ↔ 세션(등급 3) 사이에서 셋 중 무엇이 갈리는지는 표준마다 다르다 — 아래 표의 차이 열은 그래서 표준마다 다른 입력을 잰다.


| 표준 | 상시 기준(등급 1) | 세션 기준(등급 3) | 등급 1 − 등급 3 | 거리분해능 대비 |
|---|---|---|---|---|
| WiFi | 11.87 dB [^1] | 11.66 dB [^2] | +0.21 dB [^3] | 3.92 m [^17] ↔ 3.92 m [^18] |
| LTE | 13.76 dB [^19] | 13.68 dB [^20] | +0.08 dB [^8] | 16.67 m [^21] ↔ 16.64 m [^22] |
| 5G NR | 15.03 dB [^23] | 11.21 dB [^24] | +3.82 dB [^9] | 41.64 m [^25] ↔ 3.05 m [^26] |

⛔ 차이 열은 표준마다 다른 입력을 잰다 — 두 등급 사이에서 갈린 입력이 행마다 다르다. 5G 는 기준신호 대역이 7.2 MHz [^27] → 98.28 MHz [^28] 로 갈리고, LTE 는 대역이 17.985 MHz [^29] → 18.015 MHz [^30] 이고 기준신호가 CRS [^31] → PRS [^32] 로 바뀌며, WiFi 는 두 등급이 76.56 MHz [^4] 로 같다.

⚠ 데이터 점유는 송신파형 자체를 바꾸고, 이 벤치는 그 송신파형 전체를 상관 기준으로 쓴다(`src/experiment_detection.py:145`). [리포트 7 절 3 «여섯 항목은 닫힌형이고»](07_illuminators.ipynb) 는 「데이터 심볼 자체는 수신기가 내용을 몰라 정합필터 템플릿이 못 된다」 고 적었다 — 두 편의 규약이 갈린 자리이고, 어느 쪽이 야외에서 서는지는 실측 몫이다.


## 이 스윕이 서 있는 배치

이 스윕은 자유공간 배치와 **다른 배치**에서 돈다 — X410 벤치(`src/experiment_x410.py:101`), 단일 반송파 3.5 GHz [^12] 를 9모드 전부에 쓰고, 바이스태틱 거리 22.3 m [^13], 고정 σ -27.58 dBsm [^14] 다.

표적·기하·σ 를 한 값으로 묶었으므로 여기서 읽는 것은 **파형 축 하나**의 상대 비교다. 그래서 이 순서는 σ 논의가 어떻게 끝나든 그대로 남는다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 파형·수신소자 스윕을 자유공간 배치와 물리 PRF 로 옮긴다 | 이 축의 절대 SNR 이 자유공간 거리와 같은 축에 놓인다 | `src/experiment_detection.py` 의 X410Scenario → `src/freespace_scene.py` |
| 등급 2(세션 일부) 를 같은 표에 넣는다 | 상시와 풀로드 사이의 중간 체제가 대가 축에서 어디에 앉는지가 확정된다 | [리포트 7 절 3 «여섯 항목은 닫힌형이고»](07_illuminators.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 32개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report05_derived.json` | `always_on_cost.W.snr50_g1` | 11.87 |
| [^2] | `outputs/report05_derived.json` | `always_on_cost.W.snr50_g3` | 11.66 |
| [^3] | `outputs/report05_derived.json` | `always_on_cost.W.cost_db` | 0.2122 |
| [^4] | `outputs/detection_rx_sweep.json` | `modes.W1.ref_bw_mhz` | 76.56 |
| [^5] | `outputs/detection_rx_sweep.json` | `modes.W3.ref_bw_mhz` | 76.56 |
| [^6] | `outputs/detection_rx_sweep.json` | `modes.W1.occupancy` | 0.09135 |
| [^7] | `outputs/detection_rx_sweep.json` | `modes.W3.occupancy` | 0.8912 |
| [^8] | `outputs/report05_derived.json` | `always_on_cost.L.cost_db` | 0.07535 |
| [^9] | `outputs/report05_derived.json` | `always_on_cost.G.cost_db` | 3.817 |
| [^10] | `outputs/report05_derived.json` | `rx_gain.K` | 6000 |
| [^11] | `outputs/report05_derived.json` | `rx_gain.snr50_mc_sigma_db` | 0.04279 |
| [^12] | `outputs/report05_derived.json` | `bench.fc_ghz` | 3.5 |
| [^13] | `outputs/report05_derived.json` | `bench.Rb_m` | 22.28 |
| [^14] | `outputs/report05_derived.json` | `bench.sigma_dbsm` | -27.58 |
| [^15] | `outputs/report05_derived.json` | `runtime.total_h` | 7.773 |
| [^16] | `outputs/report13_freespace.json` | `meta.gpus` | 2 |
| [^17] | `outputs/report05_derived.json` | `always_on_cost.W.dr_g1_m` | 3.916 |
| [^18] | `outputs/report05_derived.json` | `always_on_cost.W.dr_g3_m` | 3.916 |
| [^19] | `outputs/report05_derived.json` | `always_on_cost.L.snr50_g1` | 13.76 |
| [^20] | `outputs/report05_derived.json` | `always_on_cost.L.snr50_g3` | 13.68 |
| [^21] | `outputs/report05_derived.json` | `always_on_cost.L.dr_g1_m` | 16.67 |
| [^22] | `outputs/report05_derived.json` | `always_on_cost.L.dr_g3_m` | 16.64 |
| [^23] | `outputs/report05_derived.json` | `always_on_cost.G.snr50_g1` | 15.03 |
| [^24] | `outputs/report05_derived.json` | `always_on_cost.G.snr50_g3` | 11.21 |
| [^25] | `outputs/report05_derived.json` | `always_on_cost.G.dr_g1_m` | 41.64 |
| [^26] | `outputs/report05_derived.json` | `always_on_cost.G.dr_g3_m` | 3.05 |
| [^27] | `outputs/detection_rx_sweep.json` | `modes.G1.ref_bw_mhz` | 7.2 |
| [^28] | `outputs/detection_rx_sweep.json` | `modes.G3.ref_bw_mhz` | 98.28 |
| [^29] | `outputs/detection_rx_sweep.json` | `modes.L1.ref_bw_mhz` | 17.98 |
| [^30] | `outputs/detection_rx_sweep.json` | `modes.L3.ref_bw_mhz` | 18.02 |
| [^31] | `outputs/detection_rx_sweep.json` | `modes.L1.ref_name` | CRS |
| [^32] | `outputs/detection_rx_sweep.json` | `modes.L3.ref_name` | PRS |


---

## 절 2. 평판·큐브·우리 격자를 같은 동작점에서 갈아끼우면 요구 이득이 이만큼 달라진다



> ### 한 일
> **같은 기하·같은 검출기·같은 동작점에서 표적만 세 모형으로 갈아끼우고, 자세평균을 맞춘 뒤 남는 요구 추가이득을 추정량별로 적었다.**

### 결과
1. 세 모형은 자세무관 평판 σ -12.81 dBsm [^33] (3GPP TR 38.901 RCS model 1 의 σ_M 상수, M1 — 확률항 σ_S 는 우리가 평균에 얼렸다) · 정육면체(M2) · 우리 SBR+PO 격자(M3, ⚠ 판: 2026-08-04 [^34] 형상 정정 전 메쉬 · 2026-08-07 10:58:22 [^35] Γ(θ) 이전 커널) 다.
2. 자세 앙상블은 셀당 1080 [^36]자세 전수, (기체×밴드) 셀은 21 [^37]개이고 재현편차는 0.00 dB [^38] 다.
3. 낙차의 소유자는 정육면체다 — 자유공간에서 최대가 M2 인 셀이 21 [^39]개, 최소가 M1 인 셀이 21 [^40]개로 전수다.
4. 크기는 **추정량이 정한다** — 검출기가 읽는 p10 에서 맞추면 낙차가 2.30 dB [^41] 로 줄어든다. ⛔ 이때 M1 이 «가장 어려운 팔» 로 올라서는 것은 M1 의 자세분산이 0.0 dB [^42] 라 어느 분위수로 맞춰도 M1 이 같은 값인 산술이고, p10 순서 계수는 `M3<M2<M1` 10 [^43]셀 · `M2<M3<M1` 11 [^44]셀 둘뿐이다.
5. 각 다양성이 낙차를 줄인다 — 각 다양성이 0 인 자유공간에서 29.30 dB [^45], 가장 큰 앙상블에서 10.09 dB [^46] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 갈아끼우는 것 | 기하·검출기·동작점을 고정하고 표적 모형만 셋으로 바꾼다 — 그래서 남는 차이는 표적 모형의 몫이다 |
| 자세 앙상블 | 셀당 자세 전수를 돌리고 자세평균을 맞춘 뒤 남는 **요구 추가이득**을 읽는다 |
| 추정량 | 선형평균(이 실험의 규약) · 중앙값 · dB 평균 · p10(검출기가 읽는 분위수) 넷을 나란히 싣는다 |
| 문턱 규약 | 잡음전력 기지 이상문턱이다. CA-CFAR 문턱은 세 팔에 같은 오프셋을 주므로 모형 간 차이는 문턱 규약에 불변이다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/tm_result.json`, `outputs/tm_attack.json` |
| 소요 | σ 격자 · 검지거리 4단계 · 검증 · 스윕을 합쳐 7.8 h [^47] (GPU 2 [^48]장). 이 빌더 자신은 CPU 수 초다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [리포트 10 절 2 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](10_results.ipynb) | 우리 격자 위의 검출거리와 순위 |
| [리포트 8 절 3 «실내 통제 기하에서 경험 Pfa 를 재니 명목…»](08_detector.ipynb) | 교정된 오경보율 위의 절대 소요이득 |

---


## 표적만 세 모형으로 갈아끼운다

같은 기하·같은 검출기·같은 동작점에서 표적만 셋으로 갈아끼운다 — 자세무관 평판 σ -12.81 dBsm [^33] (3GPP TR 38.901 RCS model 1 의 σ_M 상수, M1) · 정육면체(M2) · 우리 SBR+PO 격자(M3).

M3 팔이 선 판을 함께 찍는다 — ⚠ 판: 2026-08-04 [^34] 형상 정정 전 메쉬 · 2026-08-07 10:58:22 [^35] Γ(θ) 이전 커널. σ 격자의 신원은 「outputs/report13_sigma_grid.json (2026-07-29 생성, git 4aa668b) — 07-31 메쉬 개편(동체 셸·암 단면·블레이드) 이전 [^49]」 다. M1 은 3GPP 표값 상수이고 M2 는 현재 메쉬 bbox 로 잡은 모서리라, 이 판 표시는 M3 열에만 붙는다.

⚠ M1 의 «평평함» 은 절반만 3GPP 다 — TR 38.901 은 σ_M 과 함께 로그정규 σ_S 를 주고 그것을 경로마다 뽑는데, 헤드라인은 그것을 평균(=1)에 얼렸다. 원장이 규정대로 다시 뽑으니 M1 도 4.50 dB [^50] 를 물어 M3−M1 이 7.52 dB [^51] → 3.02 dB [^52] 로, 3모형 낙차가 29.30 dB [^53] → 24.80 dB [^54] 로 내려간다.

⛔ M1 열은 측정이 아니라 **눈금**이다 — 동작점 A_ref 를 «평판 -12.81 dBsm [^33] 이 앙상블평균 Pd = 0.9 에 정확히 앉도록» 정의했고, 원장이 그 자리에 «it is the ruler, not a result» 라고 적었다[^55].

자세 앙상블은 셀당 1080 [^36]자세 전수, (기체×밴드) 셀은 21 [^37]개이고, 자세평균을 맞춘 뒤 남는 **요구 추가이득**을 추정량별로 적는다(재현편차 0.00 dB [^38]).


| 무엇을 맞추나 | M1 평판 [dB] — 눈금(동작점 정의) | M2 정육면체 [dB] | M3 우리 SBR+PO 격자 [dB] (⚠ 판: 형상 정정 전 메쉬 · Γ(θ) 이전 커널) |
|---|---|---|---|
| 선형평균 — 이 실험의 규약 | +0.00 [^56] | +29.30 [^57] | +7.52 [^58] |
| 중앙값 | +0.00 [^59] | +16.01 [^60] | +4.78 [^61] |
| dB 평균 | +0.00 [^62] | +14.65 [^63] | +4.66 [^64] |
| p10 — 검출기가 읽는 분위수 | +0.00 [^65] | -2.20 [^66] | -2.08 [^67] |

⛔ M1 열의 네 개 +0.00 은 네 번의 독립 확인이 아니라 같은 항등식 하나다 — 원장이 재계산한 추정량 일곱 갈래(무정규화·선형평균·중앙값·dB 평균·p10·p95·최대) 전부에서 M1 이 부동소수 잡음까지 같은 값이다[^68].


## 낙차의 소유자는 정육면체다

자유공간에서 최대가 M2 인 셀이 21 [^39]개, 최소가 M1 인 셀이 21 [^40]개로 전수이고, M3 몫은 다섯 앙상블에서 20.2% [^69] ~ 33.0% [^70] 다.

각 다양성이 그 낙차를 줄인다 — 각 다양성이 0 인 자유공간(N_eff 1.0 [^71])에서 29.30 dB [^45], 각 다양성이 가장 큰 앙상블(N_eff 3.78 [^72])에서 10.09 dB [^46] 다.


## 크기는 추정량이 정한다

⚠ 낙차의 크기는 레벨을 무엇으로 맞추느냐가 정한다 — 선형평균 29.30 dB [^73] · 중앙값 16.01 dB [^74] · dB 평균 14.65 dB [^75] · p10 2.30 dB [^41] 다. 헤드라인은 선형평균 일치를 쓴다.

⛔ p10 에서 M1 이 «가장 어려운 팔» 로 올라서는 것은 분산 0 의 산술이다 — M1 의 자세분산이 0.0 dB [^42] 라 M1 은 p10 에서도 +0.00 dB [^65] 인데 M2 는 -2.20 dB [^66] · M3 는 -2.08 dB [^67] 로 내려간다. 이 정규화는 순환적이라 정본이 되지 못한다.

문턱은 잡음전력 기지 이상문턱이고 CA-CFAR 문턱은 세 팔에 같은 오프셋을 주므로, 교정표는 세 팔의 절대 소요이득만 옮긴다([^76]).


## 이 표에 구 대조군은 없다

구는 **부피를 맞게 고르면** σ 의 절대 레벨을 맞출 수 있는 단순 모형이면서 자세에 따른 변화를 0 으로 낸다. 레벨에서 우리 메쉬를 앞선 그 구는 논문이 적어 둔 상자 치수로 잡은 부피이고, 메쉬 부피로 잡으면 두 잣대 모두에서 우리 메쉬보다 나쁘다.

그래서 이 표의 낙차는 «자세 구조를 얼마나 담는가» 의 낙차로 읽는다. 구 팔을 넣는 일은 다음 단계에 있다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| M1 을 3GPP 규정대로 σ_S 를 뽑아 돌린 M1c 분기를 정본으로 세운다 | M3−M1 이 7.52 dB [^51] 인지 3.02 dB [^52] 인지가 우리 규약이 아니라 표준 규정으로 정해진다 | `scratchpad/tm_result.py` |
| 표적모형 민감도의 M3 팔을 재생성 격자(형상 정정 + Γ(θ) 켠 커널)로 다시 푼다 | 우리 팔의 요구 추가이득 7.52 dB [^77] 와 낙차 몫 25.7% [^78] 가 현재 메쉬 위에 선다 | `scratchpad/tm_result.py` |
| 표적모형에 **구 팔(M4)** 을 더한다 | 레벨만 맞추는 모형과 자세 구조를 담는 모형의 검출 낙차가 갈라진다 | `scratchpad/tm_result.py` → [리포트 3 절 4 «레벨 축에서 상자 계열은 우리 메쉬에 지고,…»](03_anchor.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 46개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^33] | `outputs/tm_result.json` | `protocol.operating_point.sigma_reference_dbsm` | -12.81 |
| [^34] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^35] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^36] | `outputs/tm_result.json` | `statistics.n_aspect_realisations_per_cell` | 1080 |
| [^37] | `outputs/tm_result.json` | `statistics.n_drone_band_cells` | 21 |
| [^38] | `outputs/tm_attack.json` | `meta.reproduction.E0_extra_gain_max_abs_dev_db` | 0 |
| [^39] | `outputs/tm_attack.json` | `Q3_staleness.argmax_argmin_counts.E0_freespace.argmax_counts.M2` | 21 |
| [^40] | `outputs/tm_attack.json` | `Q3_staleness.argmax_argmin_counts.E0_freespace.argmin_counts.M1` | 21 |
| [^41] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.p10.spread_mean` | 2.303 |
| [^42] | `outputs/tm_result.json` | `summary.E0_freespace.b_matched_mean.per_model.M1.snr_spread_std_db.max` | 0 |
| [^43] | `outputs/tm_attack.json` | `Q1_normalisation.circular_diagnostic_p10.order_counts.M3<M2<M1` | 10 |
| [^44] | `outputs/tm_attack.json` | `Q1_normalisation.circular_diagnostic_p10.order_counts.M2<M3<M1` | 11 |
| [^45] | `outputs/tm_result.json` | `verdicts.Q3_environment_dependence.pure_pattern_spread_db_by_env.E0_freespace` | 29.3 |
| [^46] | `outputs/tm_result.json` | `verdicts.Q3_environment_dependence.pure_pattern_spread_db_by_env.E2b_outdoor_shadowed` | 10.09 |
| [^47] | `outputs/report05_derived.json` | `runtime.total_h` | 7.773 |
| [^48] | `outputs/report13_freespace.json` | `meta.gpus` | 2 |
| [^49] | `outputs/tm_result.json` | `staleness.what_is_stale` | outputs/report13_sigma_grid.json (2026-07-29 생성, git 4a… |
| [^50] | `outputs/tm_attack.json` | `Q5_shape_vs_aspect_dependence.evidence[4].result.m1_penalty_if_drawn_db` | 4.503 |
| [^51] | `outputs/tm_attack.json` | `Q5_shape_vs_aspect_dependence.evidence[4].result.m3_minus_m1_frozen_db` | 7.52 |
| [^52] | `outputs/tm_attack.json` | `Q5_shape_vs_aspect_dependence.evidence[4].result.m3_minus_m1_live_db` | 3.017 |
| [^53] | `outputs/tm_attack.json` | `Q5_shape_vs_aspect_dependence.evidence[4].result.spread_frozen_mean_db` | 29.3 |
| [^54] | `outputs/tm_attack.json` | `Q5_shape_vs_aspect_dependence.evidence[4].result.spread_live_mean_db` | 24.8 |
| [^55] | `outputs/tm_result.json` | `protocol.operating_point.definition` | common link gain A_ref chosen so a FLAT target of the e… |
| [^56] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_lin.per_model_extra_gain_db.M1` | 1.275e-06 |
| [^57] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_lin.per_model_extra_gain_db.M2` | 29.3 |
| [^58] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_lin.per_model_extra_gain_db.M3` | 7.52 |
| [^59] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.median.per_model_extra_gain_db.M1` | 1.275e-06 |
| [^60] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.median.per_model_extra_gain_db.M2` | 16.01 |
| [^61] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.median.per_model_extra_gain_db.M3` | 4.784 |
| [^62] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_db.per_model_extra_gain_db.M1` | 1.275e-06 |
| [^63] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_db.per_model_extra_gain_db.M2` | 14.65 |
| [^64] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_db.per_model_extra_gain_db.M3` | 4.663 |
| [^65] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.p10.per_model_extra_gain_db.M1` | 1.275e-06 |
| [^66] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.p10.per_model_extra_gain_db.M2` | -2.199 |
| [^67] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.p10.per_model_extra_gain_db.M3` | -2.075 |
| [^68] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator` | (7항목 묶음) |
| [^69] | `outputs/tm_attack.json` | `Q3_staleness.argmax_argmin_counts.E2_outdoor_canyon.m3_share_of_spread` | 0.2016 |
| [^70] | `outputs/tm_attack.json` | `Q3_staleness.argmax_argmin_counts.E1_chamber_floor.m3_share_of_spread` | 0.3296 |
| [^71] | `outputs/tm_result.json` | `verdicts.Q3_environment_dependence.predictor.E0_freespace.n_eff_pairs` | 1 |
| [^72] | `outputs/tm_result.json` | `verdicts.Q3_environment_dependence.predictor.E2b_outdoor_shadowed.n_eff_pairs` | 3.777 |
| [^73] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_lin.spread_mean` | 29.3 |
| [^74] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.median.spread_mean` | 16.01 |
| [^75] | `outputs/tm_attack.json` | `Q1_normalisation.recomputed_spread_db_by_matching_estimator.mean_db.spread_mean` | 14.65 |
| [^76] | `outputs/tm_result.json` | `protocol.pfa_convention` | (3항목 묶음) |
| [^77] | `outputs/tm_attack.json` | `Q3_staleness.m3_own_number.base_db` | 7.52 |
| [^78] | `outputs/tm_attack.json` | `Q3_staleness.argmax_argmin_counts.E0_freespace.m3_share_of_spread` | 0.2566 |


---

## 절 3. 코히어런트 배열이득은 10log₁₀N 상한에 -0.11~+0.47 dB 로 붙는다



> ### 한 일
> **한 지점 λ/2 배열의 소자 수 N 을 늘려가며 SNR50 을 재고, 그 이득이 열잡음 코히어런트 상한 10log₁₀N 에서 얼마나 벗어나는지를 두 방식으로 확인했다.**

### 결과
1. 9모드 × N 전체에서 측정 이득은 상한 대비 -0.11 [^79] ~ +0.47 dB [^80] 다.
2. 결합 잡음전력/σ² = 0.99993 [^81] 로 잡음 보존을 확인했다 — 10log₁₀N 은 **열잡음만** 상대할 때의 이상적 상한이다.
3. 초과분의 출처는 ECA 잔차다 — 감시신호가 `surv = √N·echo + dpi + noise` 이고 `dpi` 는 N 에 무관하게 고정이라 √N 이 잡음과 잔차 양쪽 대비로 표적을 올린다.
4. 격자 보간의 대조군으로 Pd 곡선에 로지스틱을 다시 적합해도 초과분이 -0.07 [^82] ~ +0.49 dB [^83] 로 같다.
5. 몬테카를로 표준편차(K = 6000 [^84])는 0.043 dB [^85] 이고, 최대 초과분은 단일 추정 σ 의 11.0 [^86]σ · 이득(두 SNR50 추정의 차)의 √2σ 로는 7.8 σ [^87] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 배열 | 한 지점 λ/2 ULA 소자 N 개, 조향은 참 표적 방향 — `src/experiment_detection.py:181` |
| 상한의 뜻 | 10log₁₀N 은 **열잡음만** 상대할 때의 코히어런트 배열이득이고, 소자 간 결합·교정오차·위치오차가 0 인 이상적 값이다 |
| 초과분 두 방식 | 격자 보간으로 읽은 SNR50 과 로지스틱 재적합으로 읽은 SNR50 을 나란히 재, 초과분이 보간 방식의 산물인지 확인한다 |
| 잡음 보존 | 결합 잡음전력/σ² 을 재 배열 결합이 잡음을 키우거나 줄이지 않았음을 확인한다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_detection.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/detection_rx_sweep.json`, `outputs/report05_derived.json` |
| 소요 | σ 격자 · 검지거리 4단계 · 검증 · 스윕을 합쳐 7.8 h [^88] (GPU 2 [^89]장). 이 빌더 자신은 CPU 수 초다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [리포트 9 절 1 «한 순간의 (R_b»](09_observability.ipynb) | 수신기를 늘렸을 때 풀리는 것 |
| **절 1** «σ 를 곱하기 전에 이미 세 파형의 순서를 정…» | 이 스윕이 서 있는 벤치 배치 |

---


## 수신소자를 늘리면

N 은 한 지점 λ/2 ULA 소자 수다(`src/experiment_detection.py:181`). 조향벡터를 참 표적 방향에 맞추고, 결합 잡음전력/σ² = 0.99993 [^81] 로 잡음 보존을 확인했다.

그래서 10log₁₀N 은 **열잡음만** 상대할 때의 코히어런트 배열이득이고, 소자 간 결합·교정오차·위치오차가 0 인 **이상적 상한**이다.


| N | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| 측정 이득 (WiFi) = SNR50(1)−SNR50(N) | +0.00 dB | +3.37 dB | +5.24 dB | +6.44 dB |
| 열잡음 기준선 10log₁₀N | +0.00 dB | +3.01 dB | +4.77 dB | +6.02 dB |
| 차 (WiFi) | +0.00 dB | +0.36 dB | +0.47 dB | +0.42 dB |

출처 [^90]


## 초과분은 어디서 오나

9모드 × N 전체에서 측정 이득은 상한 대비 -0.11 [^79] ~ +0.47 dB [^80] 다.

감시신호가 `surv = √N·echo + dpi + noise` 이고 ECA 잔차 `dpi` 는 N 에 무관하게 고정이라(`src/experiment_detection.py:284`), √N 이 잡음과 잔차 양쪽 대비로 표적을 올린다 — x 축 SNR 은 잡음 기준 정의다(`:238`).


## 그 초과분이 보간의 산물인지 확인한다

| 검사 | 값 |
|---|---|
| Pd 곡선에 로지스틱을 다시 적합해 잰 초과분 | -0.07 [^82] ~ +0.49 dB [^83] |
| SNR50 의 몬테카를로 표준편차 (K = 6000 [^84]) | 0.043 dB [^85] |
| 최대 초과분 / 몬테카를로 σ | 11.0 [^86]σ (단일 추정 기준) · 7.8 σ [^87] (이득 = 두 추정의 차 기준) |


![fig1](../outputs/figures/report05_pf5_multirx.png)

**그림 1.** 수신소자를 늘렸을 때 얻는 감도는 이상적 코히어런트 상한에 얼마나 붙는가?


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `SIONNA2_DPI_AMP=0` 대조군으로 Rx 스윕을 다시 돌린다 | 초과분이 ECA 잔차 대비 이득임이 대조군으로 확정된다 | `src/experiment_detection.py:115` |
| 소자 간 결합과 교정오차를 넣어 상한 대비 손실을 잰다 | 이상적 상한과 실제 배열 사이의 간격이 수치로 확정된다 | **절 4** «X410 의 12-bit ADC 동적범위가 직…» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^79] | `outputs/report05_derived.json` | `rx_gain.excess_min_db` | -0.1061 |
| [^80] | `outputs/report05_derived.json` | `rx_gain.excess_max_db` | 0.4697 |
| [^81] | `outputs/detection_rx_sweep.json` | `modes.W1.combine_ratio` | 0.9999 |
| [^82] | `outputs/report05_derived.json` | `rx_gain.excess_fit_min_db` | -0.06969 |
| [^83] | `outputs/report05_derived.json` | `rx_gain.excess_fit_max_db` | 0.4875 |
| [^84] | `outputs/report05_derived.json` | `rx_gain.K` | 6000 |
| [^85] | `outputs/report05_derived.json` | `rx_gain.snr50_mc_sigma_db` | 0.04279 |
| [^86] | `outputs/report05_derived.json` | `rx_gain.excess_in_sigma` | 10.98 |
| [^87] | `outputs/report05_derived.json` | `rx_gain.excess_in_sigma → ÷√2 — 두 추정의 차` | 10.98 (파생) |
| [^88] | `outputs/report05_derived.json` | `runtime.total_h` | 7.773 |
| [^89] | `outputs/report13_freespace.json` | `meta.gpus` | 2 |
| [^90] | `outputs/detection_rx_sweep.json` | `modes.W1.curves.*.snr50` | (여러 칸) |


---

## 절 4. X410 의 12-bit ADC 동적범위가 직접파 제거의 천장이다



> ### 한 일
> **보유 장비 USRP X410 의 공식 사양을 한 곳에서 읽어 세션이 무엇에 묶이는지를 항목마다 수치로 고정했다.**

### 결과
1. 12-bit ADC(아날로그 신호를 숫자로 바꾸는 변환기)의 동적범위는 74.01 dB [^91] 이고, 이 값이 직접파 제거의 천장이다.
2. 세 파형 중 여유가 가장 좁은 것은 `LTE20` 이고 19.2 dB [^92] 다 — 점유대역이 좁아 기준채널 이득이 높다.
3. 세션은 기준 1 + 감시 1 = 2 채널 [^93] 을 같은 클럭에서 쓴다. 사양의 4 RX 는 각도축을 여는 예비다.
4. 채널당 순시대역은 400 MHz [^94] 이고 주파수 범위는 1 MHz [^95] ~ 7.2 GHz [^96] 로 세 밴드를 전부 덮는다.
5. 이 DNR 은 자유공간 시뮬 기하에서 나온 값이다 — 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 그만큼 줄어든다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사양 출처 | ni.com / ettus.com 공식 스펙 한 곳에서 인용 — `src/experiment_x410.py:61` |
| 기하 배치 | `src/experiment_x410.py:100` 한 곳에 있다 |
| 양자화 잔차 | 직접파를 양자화한 뒤 남는 잔차를 `src/experiment_x410.py:83` 의 `adc_quantize()` 가 모델에 넣는다 |
| DNR 의 출처 | 자유공간 시뮬 기하에서 계산한 값이다 — 야외 실측이 이 값을 대체한다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## X410 한 대가 기준과 감시를 동시에 든다

세션은 **RX0 = 기준(직접파)** 과 **RX1 = 감시** 두 채널을 **같은 클럭**에서 쓴다[^97] — 사양의 4 RX 는 각도축을 여는 예비다.

사양은 `src/experiment_x410.py:61`, 기하 배치는 `src/experiment_x410.py:100` 한 곳에 있다.

시뮬 사슬은 그 RX0 이 조명원의 파형 전체를 잡음도 다중경로도 없이 받는다는 상한에서 돈다. 그 가정 하나를 푼 편이 ⛔«기준채널이 현실이면 얼마를 잃는가» 별편(2026-09-03 내림 — 동작점이 실내 통제 기하다, `archive/chamber_0903/`) 이고, 이 벤치와 같은 두 채널 구성 위에서 기준채널 잡음 축을 연다.


## 사양이 무엇을 제약하나

| 항목 | 값 | 무엇을 제약하나 |
|---|---|---|
| TX / RX 채널 | 4 [^98] / 4 [^99] | 세션은 기준 1 + 감시 1 을 공통 클럭에서 쓴다 |
| 채널당 순시대역 | 400 MHz [^94] | 거리분해능과 점표적 서브밴드 |
| 주파수 범위 | 1 MHz [^95] ~ 7.2 GHz [^96] | 세 밴드 전부 커버 |
| ADC 동적범위 | 74.01 dB [^91] | 직접파 제거의 천장 |
| 감시배열 AoA 빔폭 | 33.8° [^100] | RX0 을 기준으로 두고 감시 ULA 3 소자 [^101] (RX1~3)를 쓸 때 열리는 각도축 |
| 최대대역 바이스태틱 ΔR | 0.749 m [^102] | 표적이 퍼지는 폭 |

원사양 출처는 `src/experiment_x410.py:61 (ni.com / ettus.com 2024 spec)` 한 곳이다.


## 여유가 가장 좁은 파형

![report06_adc_headroom](../outputs/figures/report06_adc_headroom.png)

**그림 2.** 12-bit ADC 는 직접파 대 잡음비 위에 얼마의 여유를 남기는가?
여유가 가장 좁은 파형은 `LTE20` 이고 19.2 dB [^92] 다 — 점유대역이 좁아 기준채널 이득이 높다.

이 DNR 은 자유공간 시뮬 기하에서 나온 값이다[^103]. 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 그만큼 줄어든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 직접파를 실제로 받아 ECA 잔차를 잰다 | 여유 19.2 dB [^92] 가 야외에 얼마나 남는지가 측정값으로 확정된다 | [리포트 1 절 3 «결정표»](01_map.ipynb) 의 사슬 확인 행 |
| 네 RX 를 전부 감시로 두는 배치를 따로 설계한다 | 각도축이 열리고 그 각도축이 검출 이후의 확장축이 된다 | `src/experiment_x410.py:100` 확장 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 13개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^91] | `outputs/report06_measurement.json` | `hw.dynamic_range_db` | 74.01 |
| [^92] | `outputs/report06_measurement.json` | `adc.headroom_db_min` | 19.23 |
| [^93] | `outputs/report06_derived.json` | `layers.n_channels` | 2 |
| [^94] | `outputs/report06_measurement.json` | `hw.max_bw_mhz` | 400 |
| [^95] | `outputs/report06_derived.json` | `hw_span.f_lo_mhz` | 1 |
| [^96] | `outputs/report06_derived.json` | `hw_span.f_hi_ghz` | 7.2 |
| [^97] | `outputs/measurement_layers.json` | `validation_three_points.channels` | reference + surveillance = 2 channels on a common clock… |
| [^98] | `outputs/report06_measurement.json` | `hw.n_tx` | 4 |
| [^99] | `outputs/report06_measurement.json` | `hw.n_rx` | 4 |
| [^100] | `outputs/report06_measurement.json` | `hw.aoa_beamwidth_deg` | 33.84 |
| [^101] | `outputs/report06_measurement.json` | `hw.n_surveillance_ch` | 3 |
| [^102] | `outputs/report06_measurement.json` | `hw.range_res_bistatic_m_at_max_bw` | 0.7495 |
| [^103] | `outputs/report06_measurement.json` | `adc.dnr_source` | outputs/verify_cfar.json:chain.<wf>.dnr_db (simulated g… |


---

## 절 5. 교정된 절대 σ 를 만드는 조건은 여섯 항목이 전부다



> ### 한 일
> **교정된 절대 σ 를 만드는 조건을 실행 항목 여섯 개로 적고 항목마다 만족해야 할 수치 임계를 붙였다.**

### 결과
1. 여섯 항목은 기준체 · 배경차감 · 자세통제 · 안테나 패턴교정 · 원거리장 · 점표적 대역이다.
2. 기준체 여유는 +3.73 dB [^104] 이상, 세션 거리는 24.44 m [^105] 이상이다.
3. 자세 표본 간격은 1.38° [^106] 이하, 점표적 서브밴드는 200 MHz [^107] 이하다.
4. 배경 차감은 지면반사 경로차가 거리분해능보다 커야 성립하고, 기하 78% [^108] 가 그 조건을 만족한다.
5. 순위 판정은 이 중 앞의 셋만 요구한다 — 여섯을 다 채운 세션이 다음 라운드의 더 센 주장(절대 σ)을 만든다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 여섯 항목의 출처 | 산문판 조건은 `docs/MEASUREMENT_PLAN.md` 에 있고, 수치 임계는 `outputs/report06_derived.json` 한 곳에서 계산된다 |
| 임계의 성격 | 왼쪽은 세션에서 **하는 일**, 오른쪽은 그 일이 만족해야 하는 **수치 임계**다 |
| 무엇이 순위 판정에 필요한가 | 앞의 셋(기준체 · 배경차감 · 자세통제)만 있으면 파형 순위는 결판난다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 교정된 절대 σ 로 가는 세션 — 실행 체크리스트

이 여섯 항목이 **교정된 절대 σ** 를 만드는 조건 전부다. 순위 판정은 이 중 앞의 셋만 요구하고, 여섯을 다 채운 세션이 다음 라운드의 더 센 주장을 만든다.

왼쪽은 세션에서 **하는 일**, 오른쪽은 그 일이 만족해야 하는 **수치 임계**다.


## 여섯 항목

| 실행 항목 | 세션에서 하는 일 | 수치 임계 |
|---|---|---|
| 교정 기준체 | 정밀 PEC 구를 세션 **시작과 끝**에 표적과 같은 지지대·같은 위치에서 잰다 | 예상 σ 대비 여유 ≥ +3.73 dB [^104] |
| 배경 차감 | 지지대를 세운 채 표적만 치우고 배경 응답을 **복소수로** 뺀다 | 지면반사 경로차 > ΔR — 기하 78% [^108] 가 분리 |
| 자세 통제 | 엔코더 턴테이블로 방위를 돌리고 로터를 정지시켜 블레이드 방위를 기록한다 | Δφ ≤ 1.38° [^106] |
| 안테나 패턴 교정 | 교정구를 표적과 같은 자리·같은 높이에 놓아 패턴과 체인 이득을 비율로 소거한다 | 표적/교정구 위치 동일 |
| 원거리장 거리 | 2D²/λ 이상에서 잰다 — 교정구는 같은 자리에 놓으면 자동 만족한다 | R ≥ 24.44 m [^105] |
| 점표적 서브밴드 | 400 MHz [^109] 순시대역을 쪼개 서브밴드마다 σ 를 내고, 그 다발을 σ(f) 로 삼는다 | B ≤ 200 MHz [^107] |


## 항목마다 자기 편이 있다

| 실행 항목 | 그 항목을 세우는 편 |
|---|---|
| 교정 기준체 · 안테나 패턴 교정 | [리포트 11 절 2 «교정 기준체»](11_measurement.ipynb) |
| 배경 차감 · 원거리장 거리 | [리포트 11 절 1 «부지 기하»](11_measurement.ipynb) |
| 자세 통제 | [리포트 11 절 4 «자세 통제»](11_measurement.ipynb) |
| 점표적 서브밴드 | [리포트 11 절 3 «점표적 서브밴드»](11_measurement.ipynb) |
| 세션을 층으로 쌓는 법 | [리포트 11 절 5 «실측 3층»](11_measurement.ipynb) |


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 기체 2종을 입고하고 이 여섯 항목대로 세션을 돌린다 | 우리 기체의 절대 σ(f, φ) 가 외부 앵커 없이 자체 측정으로 선다 | `outputs/measured_sigma.json` → `src/sigma_anchor.py:255` 앵커 등록 |
| 앞의 셋만 채운 짧은 세션을 먼저 돌린다 | 파형 상대순위 판정이 절대 σ 보다 먼저 닫힌다 | [리포트 11 절 6 «캠페인이 결판내는 양»](11_measurement.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 6개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^104] | `outputs/report06_derived.json` | `calibration_margin_min_db` | 3.732 |
| [^105] | `outputs/report06_derived.json` | `farfield_adopted.R_ff_max_m` | 24.44 |
| [^106] | `outputs/report06_derived.json` | `aspect_finest_deg` | 1.375 |
| [^107] | `outputs/report06_derived.json` | `point_target_max_bw_MHz` | 200 |
| [^108] | `outputs/report06_derived.json` | `ground_bounce_sep_frac_200MHz` | 0.7778 |
| [^109] | `outputs/report06_measurement.json` | `hw.max_bw_mhz` | 400 |
